# 00 — Preparación del corpus

**Práctica de *LLMs aplicados a Finanzas* — MIAX — Agente investigador sobre 10-K**

Este notebook es el primero que hay que ejecutar, y solo hay que ejecutarlo
una vez. Deja la carpeta `corpus/` montada en la raíz del proyecto con los
cinco ficheros de los que depende todo lo demás:

| Fichero | Qué es | Qué lo usa |
| --- | --- | --- |
| `corpus/secciones.jsonl` | el texto íntegro de las 48 secciones | `read_section` |
| `corpus/chunks.jsonl` | esas secciones troceadas en 1.749 fragmentos | `search_filings` |
| `corpus/xbrl_facts.parquet` | los hechos numéricos reportados en XBRL | `get_xbrl_fact`, el guardrail y el evaluador de cifras |
| `corpus/indice/corpus.faiss` | el índice vectorial sobre los fragmentos | la búsqueda densa |
| `corpus/indice/chunks_meta.parquet` | los metadatos alineados con el índice | la búsqueda densa y los filtros |

## Por qué este notebook existe, y por qué no existía en clase

El material de la práctica se reparte en dos ZIP: `corpus_miax_2026.zip`
(textos y hechos XBRL) e `indice_faiss.zip` (el índice). En este proyecto
**solo llegó el segundo**, ya descomprimido en `dataset/indice/`. Faltan, por
tanto, los tres primeros ficheros de la tabla.

Hay tres respuestas posibles a eso, y conviene decir por qué se elige la
tercera:

1. **Pedir el ZIP y esperar.** Es lo correcto si se puede, y el código lo
   prefiere: si `corpus_miax_2026.zip` aparece en la raíz, se usa y se
   verifica su SHA-256 como en clase. Pero no se puede condicionar la
   entrega a que aparezca un fichero.
2. **Trabajar solo con lo que hay.** Inviable: sin `xbrl_facts.parquet` no
   hay `get_xbrl_fact`, y sin `get_xbrl_fact` no hay ni middleware de
   verificación numérica ni evaluador de cifras. Son dos de los cinco
   entregables del §4 del enunciado.
3. **Reconstruirlo, y demostrar que la reconstrucción es correcta.** Es lo
   que hace este notebook. La clave está en la segunda mitad de la frase: no
   basta con producir ficheros con la pinta adecuada, hay que poder probar
   que su contenido es el original.

## De dónde sale cada pieza, y cómo se prueba que está bien

**Los textos estaban dentro del índice.** `chunks_meta.parquet` guarda, junto
a cada vector, el texto íntegro del fragmento y los desplazamientos
`inicio_car`/`fin_car` que ocupaba dentro de su sección. Escribiendo cada
fragmento en su desplazamiento se recompone la sección entera.

La prueba no es un argumento, es una comprobación. El golden set oficial
declara 13 `ancla_texto`: frases literales del informe, con el
desplazamiento exacto en el que empiezan y acaban, calculados por el profesor
sobre el texto original. Si al recortar el texto reconstruido por
`[ancla_inicio, ancla_fin)` sale exactamente esa frase —carácter a carácter,
en las 13— entonces el texto reconstruido coincide con el original en 13
puntos repartidos por 5 compañías, 2 ejercicios y 3 secciones distintas. Una
reconstrucción desplazada aunque fuera un solo carácter fallaría todas.

**Los hechos XBRL son públicos.** Son los datos que las propias compañías
presentaron ante la SEC, y la SEC los publica en una API de solo lectura,
`data.sec.gov/api/xbrl/companyfacts`. Esto **no** es lo que el enunciado
prohíbe: el aviso del §3 es sobre descargar los documentos completos de
EDGAR, que limita a 10 peticiones por segundo y bloquearía a treinta
cuadernos a la vez. Aquí son **seis peticiones en total**, a un JSON
estructurado, y se cachean en `.cache/sec/` para que a partir de la primera
ejecución el proyecto funcione sin red.

La prueba, otra vez, son dos comprobaciones y no un argumento:

- las cifras que el golden set oficial declara como verdad (12 pares
  concepto/valor) tienen que coincidir **al céntimo**;
- los huecos que el enunciado declara —Amazon sin `GrossProfit`,
  `Liabilities` ni `ResearchAndDevelopmentExpense`; Meta y Alphabet sin
  `GrossProfit`— tienen que **seguir vacíos**.

La segunda comprobación importa tanto como la primera. Una reconstrucción que
rellenara esos huecos con algo parecido rompería en silencio la parte de la
evaluación que mide si el agente sabe decir «ese dato no está en el corpus»,
que es lo que el enunciado avisa que van a preguntar al menos dos de las diez
preguntas ciegas del día 24.

In [1]:
# Dependencias. Versiones fijadas, como en los notebooks de clase.
#
# Van fijadas porque LangChain publica cada pocos días y una API que se mueve
# por debajo de un notebook lo rompe sin que se haya tocado una línea. `>=1.3`
# no es una versión. Verificado el 21/09/2026.
#
# Si ya están instaladas, esta celda tarda segundos.
%pip install -q -r requirements.txt
print("Instalación terminada.")

Note: you may need to restart the kernel to use updated packages.
Instalación terminada.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# El proyecto se importa como paquete. Añadimos la raíz al path para que
# funcione igual desde el notebook, desde una terminal o desde Colab.
import sys
from pathlib import Path

RAIZ = Path.cwd()
for carpeta in (RAIZ, RAIZ / "modulos"):
    if str(carpeta) not in sys.path:
        sys.path.insert(0, str(carpeta))

import pandas as pd

from agente import config
from agente import corpus

print("raíz del proyecto:", RAIZ)
print("índice de partida:", config.DIR_DATASET / "indice")
print("destino del corpus:", config.DIR_CORPUS)

raíz del proyecto: C:\Users\jdmar\Desktop\Taller NLP
índice de partida: C:\Users\jdmar\Desktop\Taller NLP\dataset\indice
destino del corpus: C:\Users\jdmar\Desktop\Taller NLP\corpus


## 1. Qué hay antes de empezar

Se mira lo que se tiene antes de construir nada. El objetivo de esta celda es
que quede por escrito, en la salida del notebook, qué ficheros existían de
partida: es lo que hace auditable todo lo que viene después.

In [3]:
def inventario(carpeta: Path, titulo: str) -> None:
    print(f"--- {titulo} ---")
    if not carpeta.exists():
        print("  (no existe)")
        return
    for ruta in sorted(carpeta.rglob("*")):
        if ruta.is_file():
            rel = ruta.relative_to(carpeta)
            print(f"  {str(rel):32s} {ruta.stat().st_size / 1e6:8.2f} MB")


inventario(config.DIR_DATASET, "dataset/ — lo que entregó el profesor")

for nombre in ("corpus_miax_2026.zip", "indice_faiss.zip"):
    ruta = config.RAIZ / nombre
    print(f"{nombre}: {'PRESENTE' if ruta.is_file() else 'ausente'}")

--- dataset/ — lo que entregó el profesor ---
  indice\chunks_meta.parquet           1.48 MB
  indice\corpus.faiss                  2.69 MB
  indice\MANIFEST.md                   0.00 MB
corpus_miax_2026.zip: ausente
indice_faiss.zip: ausente


## 2. La reconstrucción

`construir_corpus()` encadena cinco pasos y **se detiene con una excepción**
si cualquiera de las verificaciones falla. Eso es deliberado: un corpus mal
reconstruido que siga adelante en silencio produce métricas que parecen
correctas y no lo son, que es exactamente el modo de fallo que la sesión 2
enseña a detectar.

Los cinco pasos:

1. Si están los ZIP originales, se usan y se verifica su SHA-256. Si existe
   el dato original, se usa el original; la reconstrucción es una respuesta a
   que falte, no una mejora.
2. Se copia el índice desde `dataset/indice/` a `corpus/indice/`, que es
   donde lo buscan `miax_s1.py` y `miax_s2.py` sin modificarlos.
3. Se vuelca `chunks.jsonl` desde los metadatos del índice.
4. Se recomponen las 48 secciones y **se verifican los 13 anclas**.
5. Se descargan los hechos XBRL y **se verifican contra el golden set y
   contra los huecos declarados**.

In [4]:
resumen = corpus.construir_corpus(forzar=True)
print()
print("resumen:", resumen)

No hay ZIP originales. Reconstruyendo el corpus.
  índice copiado a C:\Users\jdmar\Desktop\Taller NLP\corpus\indice
  chunks.jsonl: 1749 fragmentos


  secciones.jsonl: 48 secciones, 13/13 anclas del golden set oficial verificadas carácter a carácter


  xbrl_facts.parquet: 137 hechos, 14 conceptos, verificados contra el golden set
  MANIFIESTO.md escrito en C:\Users\jdmar\Desktop\Taller NLP\corpus

resumen: {'origen': 'reconstruido', 'n_fragmentos': 1749, 'n_secciones': 48, 'n_anclas_verificadas': 13, 'n_hechos_xbrl': 137}


## 3. La verificación, explícita

`construir_corpus()` ya ha comprobado todo esto por dentro, pero una
comprobación que ocurre dentro de una función y no se ve no convence a nadie.
Aquí se repite a la vista, con sus números.

In [5]:
import json

secciones = corpus.cargar_secciones()
chunks = corpus.cargar_chunks()
xbrl = corpus.cargar_xbrl()
golden = [json.loads(l) for l in open(config.RUTA_GOLDEN_OFICIAL, encoding="utf-8") if l.strip()]

# --- 3.1 Alineación índice / metadatos / fragmentos -------------------------
meta = pd.read_parquet(config.RUTA_META)
import faiss

indice = faiss.read_index(str(config.RUTA_FAISS))

print("ALINEACIÓN")
print(f"  vectores en el índice FAISS : {indice.ntotal:,}")
print(f"  filas en chunks_meta.parquet: {len(meta):,}")
print(f"  líneas en chunks.jsonl      : {len(chunks):,}")
assert indice.ntotal == len(meta) == len(chunks), "Índice y fragmentos desalineados."
assert [c["chunk_id"] for c in chunks] == meta["chunk_id"].tolist(), (
    "Los chunk_id no van en el mismo orden que los vectores. El retrieval "
    "devolvería el texto equivocado sin dar ningún error."
)
print("  los chunk_id coinciden uno a uno y en el mismo orden: correcto")

ALINEACIÓN
  vectores en el índice FAISS : 1,749
  filas en chunks_meta.parquet: 1,749
  líneas en chunks.jsonl      : 1,749
  los chunk_id coinciden uno a uno y en el mismo orden: correcto


In [6]:
# --- 3.2 Los 13 anclas del golden set oficial ------------------------------
#
# Esta es la prueba fuerte de que la recomposición del texto es correcta.
problemas = corpus.verificar_anclas(secciones, golden)
con_ancla = [g for g in golden if g.get("ancla_texto")]

print(f"ANCLAS ({len(con_ancla)} preguntas del golden set oficial las llevan)")
por_clave = {(s["ticker"], s["fiscal_year"], s["item"]): s for s in secciones}
for g in con_ancla:
    seccion = por_clave[(g["ticker"], g["fiscal_year"], g["item_esperado"])]
    recorte = seccion["texto"][g["ancla_inicio"]:g["ancla_fin"]]
    marca = "OK " if recorte == g["ancla_texto"] else "MAL"
    print(f"  {marca} {g['id']}  {g['ticker']} FY{g['fiscal_year']} item "
          f"{g['item_esperado']:>2}  [{g['ancla_inicio']}, {g['ancla_fin']})")

assert not problemas, problemas
print(f"\n  {len(con_ancla)}/{len(con_ancla)} anclas caen carácter a carácter "
      f"en su desplazamiento. La reconstrucción del texto es correcta.")

ANCLAS (13 preguntas del golden set oficial las llevan)
  OK  of-001  NVDA FY2025 item 1A  [31820, 31968)
  OK  of-002  MSFT FY2025 item 1A  [19861, 19982)
  OK  of-003  META FY2025 item 1A  [130407, 130576)
  OK  of-004  AAPL FY2025 item 1A  [6147, 6237)
  OK  of-005  GOOGL FY2025 item 1A  [58086, 58260)
  OK  of-006  AMZN FY2025 item 7A  [3009, 3133)
  OK  of-014  MSFT FY2025 item  7  [1981, 2037)
  OK  of-015  NVDA FY2025 item  7  [1972, 2146)
  OK  of-016  META FY2025 item  7  [42052, 42203)
  OK  of-017  AMZN FY2025 item  7  [45866, 45992)
  OK  of-018  GOOGL FY2025 item  7  [37799, 37929)
  OK  of-019  AAPL FY2025 item  7  [7247, 7424)
  OK  of-020  META FY2025 item  7  [47337, 47482)

  13/13 anclas caen carácter a carácter en su desplazamiento. La reconstrucción del texto es correcta.


In [7]:
# --- 3.3 Los hechos XBRL contra el golden set ------------------------------
problemas_xbrl = corpus.verificar_xbrl(xbrl, golden)

print("CIFRAS DEL GOLDEN SET OFICIAL")
for g in golden:
    if not g.get("concept_xbrl") or g.get("cifra_esperada") is None:
        continue
    hecho = corpus.hecho_xbrl(g["ticker"], g["fiscal_year"], g["concept_xbrl"])
    obtenido = hecho["value"] if hecho else None
    marca = "OK " if obtenido == g["cifra_esperada"] else "MAL"
    print(f"  {marca} {g['id']}  {g['ticker']} FY{g['fiscal_year']} "
          f"{g['concept_xbrl'][:46]:46s} "
          f"reconstruido={obtenido:>18,.0f}  golden={g['cifra_esperada']:>18,.0f}")

assert not problemas_xbrl, problemas_xbrl
print("\n  Todas las cifras coinciden al céntimo.")

CIFRAS DEL GOLDEN SET OFICIAL
  OK  of-007  NVDA FY2024 Revenues                                       reconstruido=    60,922,000,000  golden=    60,922,000,000
  OK  of-008  AAPL FY2025 RevenueFromContractWithCustomerExcludingAssess reconstruido=   416,161,000,000  golden=   416,161,000,000
  OK  of-009  MSFT FY2025 NetIncomeLoss                                  reconstruido=   101,832,000,000  golden=   101,832,000,000
  OK  of-010  META FY2024 ResearchAndDevelopmentExpense                  reconstruido=    43,873,000,000  golden=    43,873,000,000
  OK  of-011  AMZN FY2025 NetCashProvidedByUsedInOperatingActivities     reconstruido=   139,514,000,000  golden=   139,514,000,000
  OK  of-012  GOOGL FY2025 Revenues                                       reconstruido=   402,836,000,000  golden=   402,836,000,000
  OK  of-013  NVDA FY2025 GrossProfit                                    reconstruido=    97,858,000,000  golden=    97,858,000,000
  OK  of-014  MSFT FY2025 RevenueFromContract

In [8]:
# --- 3.4 Los huecos declarados siguen vacíos -------------------------------
#
# El §3 del enunciado dice literalmente qué conceptos no están. Si la
# reconstrucción los rellenara, «¿cuál fue el margen bruto de Amazon?» dejaría
# de tener como respuesta correcta «no está en el corpus», y el enunciado avisa
# de que al menos dos de las diez preguntas ciegas del día 24 son de ese tipo.
print("HUECOS DECLARADOS EN EL ENUNCIADO")
for ticker, concepto in corpus.HUECOS_DECLARADOS:
    for fy in config.EJERCICIOS:
        hecho = corpus.hecho_xbrl(ticker, fy, concepto)
        marca = "OK " if hecho is None else "MAL"
        estado = "ausente (correcto)" if hecho is None else f"PRESENTE: {hecho['value']:,.0f}"
        print(f"  {marca} {ticker} FY{fy} {concepto:32s} {estado}")

print("\nY los conceptos que se han descartado a propósito para no rellenarlos "
      "por la puerta de atrás:")
for c in corpus.CONCEPTOS_DESCARTADOS:
    print(f"  - {c}")

HUECOS DECLARADOS EN EL ENUNCIADO
  OK  AMZN FY2024 GrossProfit                      ausente (correcto)
  OK  AMZN FY2025 GrossProfit                      ausente (correcto)
  OK  AMZN FY2024 Liabilities                      ausente (correcto)
  OK  AMZN FY2025 Liabilities                      ausente (correcto)
  OK  AMZN FY2024 ResearchAndDevelopmentExpense    ausente (correcto)
  OK  AMZN FY2025 ResearchAndDevelopmentExpense    ausente (correcto)
  OK  META FY2024 GrossProfit                      ausente (correcto)
  OK  META FY2025 GrossProfit                      ausente (correcto)
  OK  GOOGL FY2024 GrossProfit                      ausente (correcto)
  OK  GOOGL FY2025 GrossProfit                      ausente (correcto)

Y los conceptos que se han descartado a propósito para no rellenarlos por la puerta de atrás:
  - CostOfGoodsAndServicesSold
  - LiabilitiesAndStockholdersEquity


### Por qué se descartan `CostOfGoodsAndServicesSold` y `LiabilitiesAndStockholdersEquity`

Amazon y Microsoft sí etiquetan el coste de ventas en us-gaap. Incluirlo en
la tabla sería más completo y sería un error: teniendo los ingresos y el
coste de ventas, el margen bruto de Amazon se obtiene restando, y la
respuesta correcta a esa pregunta dejaría de ser «no está en el corpus».
Lo mismo con el total de pasivo y fondos propios, del que se despeja el
pasivo de Amazon.

La regla que se sigue, y que conviene dejar escrita porque gobierna varias
decisiones más adelante: **el corpus reconstruido puede tener menos conceptos
que la realidad, pero no puede tener menos huecos.** Los conceptos de más son
cobertura; los huecos de menos son parte de lo que se evalúa, destruido.

## 4. Paseo por los datos

Antes de optimizar nada conviene pasearse por los DataFrames. Esta sección no
construye nada: mira lo que hay, y de aquí salen varias de las decisiones de
los notebooks siguientes.

In [9]:
# --- 4.1 Anatomía del corpus: cuánto ocupa cada 10-K -----------------------
df_secciones = pd.DataFrame([{k: v for k, v in s.items() if k != "texto"} for s in secciones])

tabla = df_secciones.pivot_table(
    index=["ticker", "fiscal_year"], columns="item", values="n_tokens"
).astype(int)
tabla["TOTAL"] = tabla.sum(axis=1)
print(tabla.to_string())

print(f"\nCorpus entero: {df_secciones.n_tokens.sum():,} tokens en "
      f"{len(df_secciones)} secciones")
print(f"Informe medio: {tabla['TOTAL'].mean():,.0f} tokens")
mayor = df_secciones.nlargest(1, "n_tokens").iloc[0]
# Ojo con `mayor.item`: en pandas eso es el método Series.item, no la columna.
print(f"Sección mayor: {mayor['ticker']} FY{mayor['fiscal_year']} "
      f"Item {mayor['item']} con {mayor['n_tokens']:,} tokens")

item                   1A      7    7A      8  TOTAL
ticker fiscal_year                                  
AAPL   2024         11663   3814   612  15999  32088
       2025         11626   4294   612  16358  32890
AMZN   2024         10318   9597  1614  28097  49626
       2025         10516   9034  1547  29103  50200
GOOGL  2024         14727  11947  1877  30380  58931
       2025         14984  10648  1579  31845  59056
META   2024         33573  12621  1128  28501  75823
       2025         34751  12518  1144  32351  80764
MSFT   2024         12650  10295   409  28455  51809
       2025         11793   9510   409  26500  48212
NVDA   2024         18681   8348   635  26909  54573
       2025         19476   7824   638  27209  55147

Corpus entero: 649,119 tokens en 48 secciones
Informe medio: 54,093 tokens
Sección mayor: META FY2025 Item 1A con 34,751 tokens


Esa última línea es el argumento entero a favor de que `read_section` sea la
herramienta cara y de último recurso: una sola llamada puede meter más de
treinta mil tokens en el contexto, y se pagan en **cada** vuelta siguiente
del bucle del agente, no solo en la primera.

In [10]:
# --- 4.2 El troceado, visto con los datos ----------------------------------
#
# `contiene_tabla` marca los fragmentos donde el troceador partió una tabla. No
# es un caso raro, y eso condiciona lo que se puede esperar del retrieval sobre
# el Item 8.
df_chunks = pd.DataFrame([{k: v for k, v in c.items() if k != "texto"} for c in chunks])

resumen_chunks = df_chunks.groupby("item").agg(
    fragmentos=("chunk_id", "size"),
    con_tabla=("contiene_tabla", "sum"),
    tokens_medios=("n_tokens", "mean"),
).round(0).astype(int)
resumen_chunks["% con tabla"] = (
    100 * resumen_chunks.con_tabla / resumen_chunks.fragmentos
).round(0).astype(int)
print(resumen_chunks.to_string())
print(f"\nTotal: {int(df_chunks.contiene_tabla.sum())} de {len(df_chunks)} "
      f"fragmentos ({100 * df_chunks.contiene_tabla.mean():.0f} %) llevan tabla.")

      fragmentos  con_tabla  tokens_medios  % con tabla
item                                                   
1A           533         34            443            6
7            307        120            371           39
7A            37         18            345           49
8            872        549            390           63

Total: 721 de 1749 fragmentos (41 %) llevan tabla.


Dos lecturas de esta tabla, y las dos se usan después:

- El **Item 7A** tiene muy pocos fragmentos comparado con el resto. Una
  pregunta cuya respuesta viva ahí compite, sin filtros, contra los ~1.700
  fragmentos restantes, y nadie escribe «7A» en su pregunta. Es el primer
  fallo que arregla el filtro por metadatos del notebook 04.
- Casi la mitad de los fragmentos llevan una tabla partida dentro. Eso es lo
  que hace que leer una cifra de la prosa sea frágil aunque hoy acierte, y
  por tanto lo que justifica que `get_xbrl_fact` sea la ruta obligatoria para
  cualquier número.

In [11]:
# --- 4.3 Los conceptos XBRL, uno a uno -------------------------------------
#
# Esta es la tabla que hay que mirar antes de escribir ninguna pregunta
# numérica. El enunciado avisa: el concepto del ingreso NO es universal, y
# razonar por analogía con otra compañía es el origen del fallo «se inventó la
# cifra».
cobertura = xbrl.pivot_table(
    index="concept", columns=["ticker", "fiscal_year"], values="value", aggfunc="first"
).notna().astype(int)
cobertura = cobertura.replace({1: "si", 0: "-"})
print(cobertura.to_string())
print(f"\n{len(xbrl)} hechos · {xbrl.concept.nunique()} conceptos · "
      f"{xbrl.ticker.nunique()} compañías · unidades: "
      f"{dict(xbrl.unit.value_counts())}")

ticker                                              AAPL      AMZN      GOOGL      META      MSFT      NVDA     
fiscal_year                                         2024 2025 2024 2025  2024 2025 2024 2025 2024 2025 2024 2025
concept                                                                                                         
Assets                                                si   si   si   si    si   si   si   si   si   si   si   si
CostOfRevenue                                          -    -    -    -    si   si   si   si    -    -   si   si
EarningsPerShareDiluted                               si   si   si   si    si   si   si   si   si   si   si   si
GrossProfit                                           si   si    -    -     -    -    -    -   si   si   si   si
IncomeTaxExpenseBenefit                               si   si   si   si    si   si   si   si   si   si   si   si
Liabilities                                           si   si    -    -    si   si   si   si   s

Lo que hay que leer en esa matriz, porque es lo que decide si una pregunta
numérica está bien planteada:

- **`Revenues` contra `RevenueFromContractWithCustomerExcludingAssessedTax`.**
  NVIDIA usa la primera. Apple, Microsoft, Meta y Amazon, la segunda.
  Alphabet etiqueta las dos en FY2024 y solo la primera en FY2025. No hay
  ninguna regla que permita adivinarlo: hay que mirar el fichero.
- **Amazon no tiene `GrossProfit`, `Liabilities` ni
  `ResearchAndDevelopmentExpense`.** Meta y Alphabet tampoco `GrossProfit`.
  Son huecos reales de la taxonomía us-gaap, no fallos del corpus.
- **El capex solo está en cuatro de las seis compañías.** Amazon y NVIDIA no
  etiquetan `PaymentsToAcquirePropertyPlantAndEquipment` a nivel anual, lo
  que añade otro caso legítimo de «no está en el corpus».

El total, 137 hechos, queda en el entorno de los 135 que declara el
enunciado. No coincide exactamente porque la lista de conceptos la hemos
elegido nosotros: no sabemos cuáles eligió el profesor, y lo que importa no
es el número sino que estén todos los que el golden set necesita y que los
huecos declarados sigan siendo huecos.

In [12]:
# --- 4.4 fiscal_year no es el año de presentación --------------------------
#
# La trampa del §3. Las seis compañías cierran ejercicio en cuatro meses
# distintos, elegidos así a propósito, y el criterio que usa el corpus es el
# año en que CIERRA el ejercicio, no el año en que se presenta el informe.
cierres = (
    xbrl.groupby(["ticker", "fiscal_year"])["period_end"].max().unstack()
)
print("Fecha de cierre de ejercicio según los datos reconstruidos:")
print(cierres.to_string())
print("\nEl FY2025 de NVIDIA cerró en enero de 2025; el de Alphabet, en "
      "diciembre de 2025 y se presentó en 2026. Los dos son FY2025.")

Fecha de cierre de ejercicio según los datos reconstruidos:
fiscal_year        2024        2025
ticker                             
AAPL         2024-09-28  2025-09-27
AMZN         2024-12-31  2025-12-31
GOOGL        2024-12-31  2025-12-31
META         2024-12-31  2025-12-31
MSFT         2024-06-30  2025-06-30
NVDA         2024-01-28  2025-01-26

El FY2025 de NVIDIA cerró en enero de 2025; el de Alphabet, en diciembre de 2025 y se presentó en 2026. Los dos son FY2025.


## 5. Qué queda montado

A partir de aquí, `corpus/` existe y los notebooks siguientes pueden
ejecutarse en orden. También quedan operativos `miax_s1.py` y `miax_s2.py`
**sin modificarlos**, porque su lista `CANDIDATOS_CORPUS` ya apunta a
`Path("corpus")` y es justo donde hemos escrito todo.

In [13]:
inventario(config.DIR_CORPUS, "corpus/ — listo para usar")

print()
print("Comprobación final: los módulos de clase encuentran el corpus.")
import miax_s1
import miax_s2

print("  miax_s1.dir_corpus() ->", miax_s1.dir_corpus().resolve())
print("  miax_s2.dir_corpus() ->", miax_s2.dir_corpus().resolve())

secciones_s2, chunks_s2 = miax_s2.cargar_corpus()
assert len(secciones_s2) == 48 and len(chunks_s2) == 1749
print(f"  miax_s2.cargar_corpus() -> {len(secciones_s2)} secciones, "
      f"{len(chunks_s2)} fragmentos")
print("\n§00 listo. Siguiente: S1_Herramientas_y_Bucle_Alumno.ipynb")

--- corpus/ — listo para usar ---
  chunks.jsonl                         3.81 MB
  indice\chunks_meta.parquet           1.48 MB
  indice\corpus.faiss                  2.69 MB
  indice\MANIFEST.md                   0.00 MB
  MANIFIESTO.md                        0.00 MB
  secciones.jsonl                      3.20 MB
  xbrl_facts.parquet                   0.01 MB

Comprobación final: los módulos de clase encuentran el corpus.


  miax_s1.dir_corpus() -> C:\Users\jdmar\Desktop\Taller NLP\corpus
  miax_s2.dir_corpus() -> C:\Users\jdmar\Desktop\Taller NLP\corpus
  miax_s2.cargar_corpus() -> 48 secciones, 1749 fragmentos

§00 listo. Siguiente: S1_Herramientas_y_Bucle_Alumno.ipynb
